In [1]:
import numpy as np
import pandas as pd

from PIL import Image
from tqdm import tqdm

import os
import random
import cv2
import copy

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from tqdm import tqdm
from sklearn.metrics import cohen_kappa_score, accuracy_score

from torch.optim.lr_scheduler import StepLR
import torch.nn.functional as F
from torch.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import f1_score, recall_score

In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # for multi-GPU (DataParallel)
    
    # Critical for CuDNN reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [3]:
class DRDataset(Dataset):
    def __init__(self, images_folder, dataframe, transform=None, is_test=False):
        super().__init__()
        self.df = dataframe
        self.images_folder = images_folder
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        # Get row info
        row = self.df.iloc[index]
        image_name = str(row['image'])

        label = int(row['level']) if not self.is_test else -1

        # Handle File Path
        # Adding .jpeg if not present
        if not image_name.endswith('.jpeg'):
            image_path = os.path.join(self.images_folder, f"{image_name}.jpeg")
        else:
            image_path = os.path.join(self.images_folder, image_name)

        # Load Image
        try:
            image = Image.open(image_path).convert("RGB")
            image = np.array(image)
        except Exception as e:
            print(f"Error loading {image_path}: {e}")
            
            # Return a blank image or handle error as needed for your pipeline
            image = np.zeros((512, 512, 3), dtype=np.uint8)

        # 4. Apply Transforms (Albumentations style)
        if self.transform:
            image = self.transform(image=image)["image"]

        return image, label, image_name

In [4]:
## Config

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 5e-4
BATCH_SIZE = 64

NUM_EPOCHS = 35
NUM_WORKERS = 4
CHECKPOINT_FILE = "checkpoint.pth.tar"
PIN_MEMORY = True
SAVE_MODEL = True
LOAD_MODEL = False

# OLD:  Data augmentation for images
# train_transforms = A.Compose(
#     [
#         A.HorizontalFlip(p=0.5),
#         A.VerticalFlip(p=0.5),
#         A.RandomRotate90(p=0.5),
#         A.Blur(p=0.2),
#         A.Affine(shear=30, rotate=0, p=0.2, border_mode=cv2.BORDER_CONSTANT),
#         A.Normalize(
#             mean=[0.3199, 0.2240, 0.1609],
#             std=[0.3020, 0.2183, 0.1741],
#             max_pixel_value=255.0,
#         ),
#         ToTensorV2(),
#     ]
# )

# NEW More robust Data augmentations (Safer)
train_transforms = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=30, border_mode=cv2.BORDER_CONSTANT, p=0.5),

        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.3),

        A.GaussianBlur(blur_limit=(3, 5), p=0.2),

        A.CoarseDropout(
            num_holes_range=(1, 4), 
            hole_height_range=(1, 16), 
            hole_width_range=(1, 16), 
            p=0.1
        ),

        A.Affine(shear=15, rotate=0, p=0.2, border_mode=cv2.BORDER_CONSTANT),

        A.Normalize(
            mean=[0.3199, 0.2240, 0.1609],
            std=[0.3020, 0.2183, 0.1741],
            max_pixel_value=255.0,
        ),
        ToTensorV2(),
    ]
)

val_transforms = A.Compose(
    [
        A.Normalize(
            mean=[0.3199, 0.2240, 0.1609],
            std=[0.3020, 0.2183, 0.1741],
            max_pixel_value=255.0,
        ),
        ToTensorV2(),
    ]
)

In [5]:
def train_one_epoch(loader, model, optimizer, loss_fn, device): # Remove scaler from args
    model.train()
    summary_loss = 0
    pbar = tqdm(loader, desc="Training")
    
    for images, labels, _ in pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with autocast(device):
            outputs = model(images)
            loss = loss_fn(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        summary_loss += loss.item()
        pbar.set_postfix(loss=loss.item())
    return summary_loss / len(loader)


def validate(loader, model, loss_fn, device):
    model.eval()
    summary_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels, _ in tqdm(loader, desc="Validating"):
            images = images.to(device)
            labels = labels.to(device)

            with autocast(device):
                outputs = model(images)
                loss = loss_fn(outputs, labels)

            summary_loss += loss.item()

            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    # Metrics
    avg_loss = summary_loss / len(loader)
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')

    f1 = f1_score(all_labels, all_preds, average='macro')
    recall = recall_score(all_labels, all_preds, average='macro')

    return avg_loss, qwk, f1, recall


def evaluate_test_set(loader, model, device):
    model.eval()
    all_preds = []
    all_labels = []

    print("\n--- Starting Final Test Evaluation ---")
    with torch.no_grad():
        for images, labels, _ in tqdm(loader, desc="Testing"):
            images = images.to(device)
            labels = labels.to(device)

            with torch.amp.autocast(device_type='cuda'):
                outputs = model(images)

            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    # Metrics
    test_accuracy = accuracy_score(all_labels, all_preds)
    test_qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    test_f1 = f1_score(all_labels, all_preds, average='macro')
    test_recall = recall_score(all_labels, all_preds, average='macro')

    print(f"\n[FINAL TEST RESULTS]")
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test QWK Score: {test_qwk:.4f}")
    print(f"Test F1 (Macro): {test_f1:.4f}")
    print(f"Test Recall (Macro): {test_recall:.4f}")

    return test_accuracy, test_qwk, test_f1, test_recall

In [6]:
img_dir = r'/kaggle/input/datasets/bngtiutr/diabeticretinadataset/train_cropped_512'

train_df = pd.read_csv(r'/kaggle/input/datasets/bngtiutr/diabeticretinadataset/trainLabels_new.csv')
val_df = pd.read_csv(r'/kaggle/input/datasets/bngtiutr/diabeticretinadataset/valLabels_new.csv')
test_df = pd.read_csv(r'/kaggle/input/datasets/bngtiutr/diabeticretinadataset/testLabels_new.csv')


train_ds = DRDataset(img_dir, train_df, transform=train_transforms)
val_ds = DRDataset(img_dir, val_df, transform=val_transforms)
test_ds = DRDataset(img_dir, test_df, transform=val_transforms)

In [7]:
class_counts = np.bincount(train_df['level'])

print(class_counts)

class_weights = 1.0 / np.sqrt(class_counts)
class_weights = class_weights / class_weights.sum() * len(class_counts)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print(class_weights)

[20641  1950  4230   698   567]
tensor([0.2788, 0.9071, 0.6159, 1.5161, 1.6822], device='cuda:0')


In [8]:
class WeightedCEWithOrdinalLoss(nn.Module):
    def __init__(self, class_weights=None, num_classes=5, lambda_ord=0.5, label_smoothing=0.1):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
        self.lambda_ord = lambda_ord
        self.num_classes = num_classes

        # tensor [0,1,2,...,C-1]
        self.register_buffer("class_indices", torch.arange(num_classes).float())

    def forward(self, logits, targets):
        # --- Cross Entropy ---
        ce_loss = self.ce(logits, targets)

        # --- Ordinal Loss ---
        probs = F.softmax(logits, dim=1)

        # expected value: E[y_hat]
        class_indices = self.class_indices.to(probs.device)
        pred_value = torch.sum(probs * class_indices, dim=1)

        targets = targets.float()

        ordinal_loss = torch.mean((pred_value - targets) ** 2)

        # --- Total ---
        total_loss = ce_loss + self.lambda_ord * ordinal_loss

        return total_loss

In [9]:
# Loaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=True
)

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=True
)

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=PIN_MEMORY, persistent_workers=True)

# Load timm model
import timm
model = timm.create_model(
    'resnet50.a3_in1k',
    pretrained=True,
    num_classes=5
)

model = torch.nn.DataParallel(model) 
model = model.to(DEVICE)

# Loss, Optimizer, Scaler
# OLD:  loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
loss_fn = WeightedCEWithOrdinalLoss(
    class_weights=class_weights,
    num_classes=5,
    lambda_ord=0.7,   # tune this (0.3–1.0)
    label_smoothing=0.1
)

optimizer = torch.optim.Adam(model.parameters(),
                             lr=LEARNING_RATE,
                             weight_decay=WEIGHT_DECAY)

scheduler = StepLR(optimizer, step_size=6, gamma=0.75)

scaler = GradScaler()


# --- Loop ---
best_qwk = -1
best_model_wts = copy.deepcopy(model.state_dict()) # Initialize with starting weights

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

    # Get current LR for logging
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr:.6f}")
    
    train_loss = train_one_epoch(train_loader, model, optimizer, loss_fn, DEVICE)
    val_loss, val_qwk, val_f1, val_recall = validate(val_loader, model, loss_fn, DEVICE)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val QWK: {val_qwk:.4f} | Val F1: {val_f1:.4f} | Val Recall: {val_recall:.4f}")
    
    # --- Step the scheduler here ---
    scheduler.step()
    
    # Save best model based on QWK
    if val_qwk > best_qwk:
        best_qwk = val_qwk

        # Deep copy the weights so they don't change during subsequent epochs
        best_model_wts = copy.deepcopy(model.state_dict())
        
        torch.save(model.module.state_dict(), CHECKPOINT_FILE)
        print("=> Best model saved!")

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]


Epoch 1/35
Current Learning Rate: 0.000100


Validating: 100%|██████████| 55/55 [00:32<00:00,  1.68it/s]


Train Loss: 2.5466 | Val Loss: 2.3181 | Val QWK: -0.0005 | Val F1: 0.1694 | Val Recall: 0.1999
=> Best model saved!

Epoch 2/35
Current Learning Rate: 0.000100


Validating: 100%|██████████| 55/55 [00:31<00:00,  1.77it/s]


...

Train Loss: 1.3401 | Val Loss: 1.7460 | Val QWK: 0.7736 | Val F1: 0.5970 | Val Recall: 0.5664

Epoch 34/35
Current Learning Rate: 0.000024


Validating: 100%|██████████| 55/55 [00:32<00:00,  1.70it/s]


Train Loss: 1.3263 | Val Loss: 1.7732 | Val QWK: 0.7626 | Val F1: 0.5905 | Val Recall: 0.5672

Epoch 35/35
Current Learning Rate: 0.000024


Validating: 100%|██████████| 55/55 [00:31<00:00,  1.73it/s]

Train Loss: 1.3137 | Val Loss: 1.7786 | Val QWK: 0.7625 | Val F1: 0.5804 | Val Recall: 0.5382


In [10]:
# Final Evaluation
test_acc, test_qwk, test_f1, test_recall = evaluate_test_set(test_loader, model, DEVICE)

# Save latest model to disk
LATEST_CHECKPOINT = "latest_model.pth"
torch.save(model.module.state_dict(), LATEST_CHECKPOINT)
print(f"=> Latest model weights saved to {LATEST_CHECKPOINT}")


--- Starting Final Test Evaluation ---


Testing: 100%|██████████| 55/55 [00:45<00:00,  1.22it/s]


[FINAL TEST RESULTS]
Test Accuracy: 0.8277
Test QWK Score: 0.7544
Test F1 (Macro): 0.5719
Test Recall (Macro): 0.5225
=> Latest model weights saved to latest_model.pth


In [11]:
# Load the best weights back into the model object before final evaluation
model.load_state_dict(best_model_wts)

# Final Evaluation
test_acc, test_qwk, test_f1, test_recall = evaluate_test_set(test_loader, model, DEVICE)


--- Starting Final Test Evaluation ---


Testing: 100%|██████████| 55/55 [00:32<00:00,  1.67it/s]


[FINAL TEST RESULTS]
Test Accuracy: 0.8300
Test QWK Score: 0.7742
Test F1 (Macro): 0.6001
Test Recall (Macro): 0.5896
